In [1]:
import numpy as np
import pandas as pd
import torch

In [2]:
df = pd.read_csv("/Users/teljirisahith/Desktop/dataset/talking.csv")

print(df.head())

   time_sec     FP1
0     0.000  -1.396
1     0.004   5.753
2     0.008  23.159
3     0.012  12.970
4     0.016  20.874


In [3]:
signal = df["FP1"].values

print(signal.shape)

(7500,)


In [4]:
WINDOW_SIZE = 250

windows = []

for i in range(0, len(signal), WINDOW_SIZE):

    window = signal[i:i+WINDOW_SIZE]

    if len(window) == WINDOW_SIZE:
        windows.append(window)

windows = np.array(windows)

print(windows.shape)

(30, 250)


In [5]:
for i in range(len(windows)):

    mean = np.mean(windows[i])
    std = np.std(windows[i])

    windows[i] = (windows[i]-mean)/(std+1e-8)

In [6]:
X = torch.tensor(windows, dtype=torch.float32)

X = X.unsqueeze(1)

print(X.shape)

torch.Size([30, 1, 250])


In [7]:
import torch
import torch.nn as nn

class AttentionCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
model = AttentionCNN()   # Replace CNN with your class name if it's different
model.load_state_dict(torch.load("/Users/teljirisahith/Desktop/dataset/attention_cnn.pth", map_location="cpu"))
model.eval()

AttentionCNN(
  (features): Sequential(
    (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [9]:
device = torch.device("mps")

model.to(device)
X = X.to(device)

In [10]:
with torch.no_grad():

    outputs = model(X)

    probs = torch.sigmoid(outputs)

    preds = (probs >= 0.70).int()

In [11]:
for i in range(len(preds)):
    probability = probs[i].item()

    if preds[i].item() == 1:
        label = "Attentive"
    else:
        label = "Inattentive"

    print(f"Second {i+1}: {label} (Probability: {probability:.3f})")

Second 1: Attentive (Probability: 0.978)
Second 2: Attentive (Probability: 0.991)
Second 3: Attentive (Probability: 0.981)
Second 4: Attentive (Probability: 0.959)
Second 5: Attentive (Probability: 0.994)
Second 6: Attentive (Probability: 0.984)
Second 7: Attentive (Probability: 0.987)
Second 8: Attentive (Probability: 0.985)
Second 9: Attentive (Probability: 0.946)
Second 10: Attentive (Probability: 0.976)
Second 11: Attentive (Probability: 0.948)
Second 12: Attentive (Probability: 0.996)
Second 13: Attentive (Probability: 0.989)
Second 14: Attentive (Probability: 0.976)
Second 15: Attentive (Probability: 0.985)
Second 16: Attentive (Probability: 0.949)
Second 17: Attentive (Probability: 0.991)
Second 18: Attentive (Probability: 0.963)
Second 19: Attentive (Probability: 0.980)
Second 20: Attentive (Probability: 0.978)
Second 21: Attentive (Probability: 0.995)
Second 22: Attentive (Probability: 0.974)
Second 23: Attentive (Probability: 0.977)
Second 24: Attentive (Probability: 0.978)
S